# Website Summarizer for JS heavy Websites
---

## Purpose 

In day 1 of week 1, we saw a website summarizer that uses beautiful soup for scraping websites that were not JS heavy and had content primarily in HTML. An additonal exersice of making a website summarizer was suggested -- this notebook is the work for that exercise.

### Dependencies

```python
import os
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

from openai import OpenAI

from playwright.async_api import async_playwright
```

---

## Strategy

Currently, the Website class contains the core of processing the scraping aspect of the website summarizer:

```python
class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)
```

Additional functionality will be added to the Website class to use playwright for scraping the URL of the website with JS heavy content when the beatifulsoup fails to return anything meaningful

```python
class Website:

    def __init__(self,url):
        ...

    def _scrape_with_beautifulsoup(self):
        ...
    
    async def _scrape_with_playwright(self):
        ...

    async def scrape(self):
        if self._scrape_with_beautifulsoup():
            return 

        await self._scrape_with_playwright()
```

---

## Implementation

### Section 0 - Dependencies

```bash

# install playwright
uv add playwright
uv run playwright install chromium

# verify playwright install
uv run python -c "from playwright.sync_api import sync_playwright; print('Playwright OK')"
```


### Section 1 - Imports


In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# from playwright.sync_api import sync_playwright # sync API doesnt work in Jupyter Notebook, so we need to use the async API
from playwright.async_api import async_playwright

### Section 2 - Register local Ollama model with OpenAI handler

In [ ]:
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


### Section 3 - Setup the Website class

In [ ]:
# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# a class to represent a website and handle its contents
class Website:

    MIN_TEXT_LENGTH = 500

    def __init__(self, url):
        self.url = url
        self.title = "No title found"
        self.text = ""
        self.source = None

    def _scrape_with_beautifulsoup(self):
        """
        Try to retrieve and extract the website using requests + BeautifulSoup.

        Returns True if we got enough useful text.
        Returns False if the page appears to require JavaScript rendering.
        """

        try:
            response = requests.get(
                self.url,
                headers=headers,
                timeout=15
            )

            response.raise_for_status()

            soup = BeautifulSoup(
                response.content,
                "html.parser"
            )

            # Get title
            if soup.title:
                self.title = soup.title.get_text(strip=True)

            # Make sure body exists
            if not soup.body:
                return False

            # Remove things that aren't useful to the LLM
            for irrelevant in soup.body(
                ["script", "style", "img", "input", "noscript"]
            ):
                irrelevant.decompose()

            text = soup.body.get_text(
                separator="\n",
                strip=True
            )

            # Clean up excessive blank lines
            lines = [
                line.strip()
                for line in text.splitlines()
                if line.strip()
            ]

            text = "\n".join(lines)

            # Determine whether we actually got useful content
            if len(text) < self.MIN_TEXT_LENGTH:
                return False

            self.text = text
            self.source = "beautifulsoup"

            return True

        except requests.RequestException as e:
            print(f"Requests failed: {e}")
            return False

        except Exception as e:
            print(f"BeautifulSoup extraction failed: {e}")
            return False

    async def _scrape_with_playwright(self):
        """
        Render the page using Chromium/Playwright and extract
        the resulting DOM text.
        """

        try:
            async with async_playwright() as p:

                browser = await p.chromium.launch(
                    headless=True
                )

                page = await browser.new_page(
                    user_agent=headers["User-Agent"]
                )

                await page.goto(
                    self.url,
                    wait_until="domcontentloaded",
                    timeout=30000
                )

                # Give JavaScript applications some time to render.
                await page.wait_for_timeout(2000)

                self.title = await page.title()

                # Remove elements we don't want to send to the LLM
                await page.locator(
                    "script, style, noscript, img, input"
                ).evaluate_all(
                    """elements => elements.forEach(element => element.remove())"""
                )

                self.text = await page.locator("body").inner_text()

                self.source = "playwright"

                await browser.close()

        except Exception as e:
            print(f"Playwright failed: {e}")
            self.text = ""
            self.source = None
    
    async def scrape(self):

        # Try beautifulsoup first
        if self._scrape_with_beautifulsoup():
            return
        
        print("BeautifulSoup extraction was insufficient.")
        print("Falling back to Playwright...")

        # If beautifulsoup fails, try playwright
        await self._scrape_with_playwright()


In [ ]:
# scrape a non-JS website

url = "https://www.cnn.com"

cnn_website = Website(url)

await cnn_website.scrape()

print(f"Scraped {cnn_website.url} using {cnn_website.source}. Title: {cnn_website.title}")
print(f"Text length: {len(cnn_website.text)} characters")

### Section 3 - Scrape a JS heavy website


In [ ]:
url = "https://www.openai.com"

openai_website = Website(url)

await openai_website.scrape()

print(f"Scraped {openai_website.url} using {openai_website.source}. Title: {openai_website.title}")
print(f"Text length: {len(openai_website.text)} characters")

### Section 4 - Setup the prompt for Ollama

In [ ]:
# system prompt for the LLM to summarize the website content

system_prompt = "You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown."

In [ ]:
# user prompt containing the website text to summarize

def user_prompt_for(website: Website) -> str:
    return f"""
    Please summarize the following website content. If it includes news or announcements, then summarize these too. 
    Provide a short summary in markdown format. 

    Website URL: {website.url}
    Website Title: {website.title}
    Website Text: {website.text}
    """

In [ ]:
# show user prompt

print(user_prompt_for(openai_website))

### Section 5 - Setup messages item for calling the website summarizer

In [ ]:
def messages_for(website: Website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [ ]:
messages_for_openai_website = messages_for(openai_website)
print(messages_for_openai_website)

### Section 6 - Setup the summarize and display summary function

In [ ]:
async def summarize_website_and_display_summary(url: str):
    website = Website(url)
    await website.scrape()

    messages = messages_for(website)

    response = openai.chat.completions.create(
        model="llama3.2",
        messages=messages,
        max_tokens=500
    )

    summary = response.choices[0].message.content
    
    display(Markdown(summary))

### Section 7 - Summarize Websites

In [ ]:
# Example usage:
await summarize_website_and_display_summary("https://www.cnn.com")

In [ ]:
# Example usage:
await summarize_website_and_display_summary("https://www.openai.com")